In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Initialize the Spark Session
# "local[*]" tells Spark to use all available CPU cores on your machine
spark = SparkSession.builder \
    .appName("GeoPulse_Local_Test") \
    .master("local[*]") \
    .getOrCreate()

# 2. Create some raw mock data (DeviceID, Lat, Long, Timestamp)
mock_data = [
    ("Device_123", 37.7749, -122.4194, "08:00"),
    ("Device_456", 37.7749, -122.4194, "08:05"),
    ("Device_123", 37.7750, -122.4190, "08:10"),
    ("Device_789", 34.0522, -118.2437, "08:15"),
    ("Device_123", 37.7755, -122.4180, "08:20")
]
columns = ["DeviceID", "Lat", "Long", "Time"]

# 3. Load the data into a distributed DataFrame
df = spark.read.csv(
    "Data/bengaluru_test_1m.csv",
    header=True,       # Uses the first row as column names
    inferSchema=True   # Automatically detects integers, doubles, and timestamps
)

print("\n--- Raw GPS Pings ---")
df.show()

# 4. Perform a transformation: Count how many times each device pinged
ping_counts = df.groupBy("DeviceID") \
    .agg(F.count("*").alias("Total_Pings")) \
    .orderBy(F.col("Total_Pings").desc())

print("\n--- Aggregated Ping Counts ---")
# 5. Action: spark doesn't actually compute anything until you call an action like show()
ping_counts.show()

# 6. Shut down the cluster
spark.stop()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/Data/bengaluru_test_1m.csv. SQLSTATE: 42K03

In [1]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from sedona.spark import SedonaContext

def analyze_store_traffic_distributed(csv_path, shop_lat, shop_lon, catchment_radius=500, visit_radius=20, min_pings=3):
    # 1. Initialize Spark with Sedona Spatial capabilities
    spark = SparkSession.builder \
        .appName("GeoPulse_Sedona_Catchment") \
        .config("spark.jars.packages", 
                "org.apache.sedona:sedona-spark-shaded-3.0_2.12:1.5.0,"
                "org.datasyslab:geotools-wrapper:1.5.0-28.2") \
        .getOrCreate()
    
    # Register Sedona's ST_* functions
    sedona = SedonaContext.create(spark)

    print("Loading distributed data...")
    # Load raw GPS CSV
    raw_df = sedona.read.csv(csv_path, header=True, inferSchema=True)
    
    # 2. Spatial Transformation: Convert Lat/Lon into Sedona Geometry Points
    # Note: ST_Point takes (Longitude, Latitude)
    spatial_df = raw_df.withColumn(
        "device_point", 
        F.expr("ST_Point(CAST(Longitude AS Decimal(24,20)), CAST(Latitude AS Decimal(24,20)))")
    ).withColumn(
        "shop_point", 
        F.expr(f"ST_Point(CAST({shop_lon} AS Decimal(24,20)), CAST({shop_lat} AS Decimal(24,20)))")
    )

    # 3. Calculate distance in meters using the spherical Earth model
    dist_df = spatial_df.withColumn(
        "distance_m", 
        F.expr("ST_DistanceSphere(device_point, shop_point)")
    )

    # --- METRIC 1: Broad Catchment (Unique devices in the neighborhood) ---
    catchment_df = dist_df.filter(F.col("distance_m") <= catchment_radius)
    unique_catchment_devices = catchment_df.select("DeviceID").distinct().count()

    # --- METRIC 2: Tight Visit (Inside the shop) ---
    visit_df = dist_df.filter(F.col("distance_m") <= visit_radius)
    
    # Group by DeviceID AND a 1-hour rolling time window to create "sessions"
    session_df = visit_df.groupBy(
        "DeviceID", 
        F.window("Timestamp", "1 hour")
    ).agg(
        F.count("*").alias("ping_count")
    )

    # Filter out drive-bys: Only keep sessions meeting the minimum ping threshold
    qualified_visits_df = session_df.filter(F.col("ping_count") >= min_pings)

    # --- Final Aggregations ---
    # Since qualified_visits_df contains one row per valid visit:
    total_qualified_visits = qualified_visits_df.count()
    
    # Distinct devices that had at least one qualified visit
    actual_unique_customers = qualified_visits_df.select("DeviceID").distinct().count()
    
    # Calculate the average pings per qualified visit
    if total_qualified_visits > 0:
        avg_pings_row = qualified_visits_df.select(F.avg("ping_count").alias("avg")).first()
        average_pings = round(avg_pings_row["avg"], 1)
    else:
        average_pings = 0

    spark.stop()

    return {
        "catchment_radius_m": catchment_radius,
        "unique_devices_in_catchment": unique_catchment_devices,
        "actual_unique_customers": actual_unique_customers,
        "total_visits_including_repeats": total_qualified_visits,
        "average_pings_per_visit": average_pings
    }

metrics = analyze_store_traffic_distributed("bengaluru_test_1m.csv", 37.7749, -122.4194)
print(metrics)